In [4]:
# Prediction card - fill by hand BEFORE running anything
vram_prediction = "about the same"
tokps_prediction = "faster"
tool_call_success_out_of_8 = 7

prediction = {
    "vram_prediction": vram_prediction,
    "tokps_prediction": tokps_prediction,
    "tool_call_success_out_of_8": tool_call_success_out_of_8,
}
print(prediction)

{'vram_prediction': 'about the same', 'tokps_prediction': 'faster', 'tool_call_success_out_of_8': 7}


In [3]:
!sudo apt-get update -y
!sudo apt-get install -y python3.10 python3.10-venv python3.10-dev
!python3.10 -m venv /content/venv
!/content/venv/bin/python -m pip install --upgrade pip

VENV_PYTHON = "/content/venv/bin/python"

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [5]:
import subprocess, sys

VLLM_PIN = "0.6.*"
TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"
AUTOAWQ_PIN = "0.2.*"
HTTPX_PIN = "0.27.*"
OPENAI_PIN = "1.54.*"

def pip_install(*specs):
    cmd = [VENV_PYTHON, "-m", "pip", "install", "-q", *specs]
    print("installing (venv):", " ".join(specs))
    subprocess.run(cmd, check=True)

pip_install(
    f"vllm=={VLLM_PIN}",
    f"transformers=={TRANSFORMERS_PIN}",
    f"accelerate=={ACCELERATE_PIN}",
    f"autoawq=={AUTOAWQ_PIN}",
    f"httpx=={HTTPX_PIN}",
    f"openai=={OPENAI_PIN}",
)
print("serving pins installed inside the venv (with AWQ)")


subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                f"httpx=={HTTPX_PIN}", f"openai=={OPENAI_PIN}"], check=True)
print("client-side pins installed in the notebook kernel")

installing (venv): vllm==0.6.* transformers==4.46.* accelerate==1.1.* autoawq==0.2.* httpx==0.27.* openai==1.54.*
serving pins installed inside the venv (with AWQ)
client-side pins installed in the notebook kernel


In [6]:
!/content/venv/bin/python -c "import torch; print('cuda available:', torch.cuda.is_available())"

cuda available: True


In [7]:
import os, signal, subprocess

PORT = 8000
SERVER_LOG = "/content/server.log"

SERVER_ARGS = {
    "--model": "Qwen/Qwen2.5-1.5B-Instruct-AWQ",
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": str(PORT),
    "--quantization": "awq",
    "--enable-auto-tool-choice": None,
    "--tool-call-parser": "hermes",
}

def build_cmd(args: dict) -> list:
    cmd = [VENV_PYTHON, "-m", "vllm.entrypoints.openai.api_server"]
    for k, v in args.items():
        cmd += [k] if v is None else [k, str(v)]
    return cmd

def launch_server(args: dict = None):
    args = SERVER_ARGS if args is None else args
    cmd = build_cmd(args)
    print("launching:", " ".join(cmd))
    logf = open(SERVER_LOG, "wb")
    proc = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT,
                            start_new_session=True)
    print(f"server pid {proc.pid}, logging to {SERVER_LOG}")
    return proc

server = launch_server()

launching: /content/venv/bin/python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct-AWQ --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000 --quantization awq --enable-auto-tool-choice --tool-call-parser hermes
server pid 4848, logging to /content/server.log


In [8]:
import time, urllib.request, urllib.error

def tail_log(path=SERVER_LOG, n=30):
    try:
        with open(path, "r", errors="replace") as fh:
            lines = fh.readlines()
        return "".join(lines[-n:])
    except FileNotFoundError:
        return "(no log file yet)"

def wait_for_health(port=PORT, timeout_s=300, interval_s=3):
    url = f"http://localhost:{port}/v1/models"
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    waited = int(timeout_s - (deadline - time.time()))
                    print(f"server healthy after about {waited}s: {url} -> 200")
                    return True
        except (urllib.error.URLError, ConnectionError, OSError):
            pass
        time.sleep(interval_s)
    print(f"TIMED OUT after {timeout_s}s waiting for {url}")
    print("last 30 log lines:")
    print(tail_log())
    return False

healthy = wait_for_health()

server healthy after about 60s: http://localhost:8000/v1/models -> 200


In [1]:
!nvidia-smi

Wed Sep  2 10:56:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
!nvidia-smi --query-gpu=memory.used --format=csv,noheader

11723 MiB


In [10]:
!grep -i "GPU blocks" /content/server.log

INFO 09-02 11:04:40 gpu_executor.py:76] # GPU blocks: 22955, # CPU blocks: 9362


In [11]:
import time
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="not-needed")

def measure_tokps(prompt="Explain what an inference server does in detail.", max_tokens=200):
    t0 = time.time()
    r = client.chat.completions.create(
        model="Qwen/Qwen2.5-1.5B-Instruct-AWQ",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=0.0,
    )
    dt = time.time() - t0
    ct = r.usage.completion_tokens
    tokps = round(ct / dt, 1)
    print(f"completion_tokens={ct}, wall_s={round(dt,2)}, tokens/s={tokps}")
    return tokps

awq_tokps = measure_tokps()

completion_tokens=200, wall_s=2.21, tokens/s=90.4


In [12]:
SPOT_PROMPTS = [
    "Write a two-sentence summary of what an inference server does.",
    "A user asks for the weather in Riyadh and the time in Tokyo. "
    "What two tool calls would you make?",
    "Refactor this into a single sentence: The GPU was busy but not "
    "productive, because decode is memory-bound.",
    "List the steps to roll back a bad deployment, in order.",
    "Explain quantisation to a non-technical manager in three sentences.",
]

for p in SPOT_PROMPTS:
    r = client.chat.completions.create(
        model="Qwen/Qwen2.5-1.5B-Instruct-AWQ",
        messages=[{"role": "user", "content": p}], max_tokens=200)
    print("PROMPT:", p[:50], "...")
    print(r.choices[0].message.content, "\n")

PROMPT: Write a two-sentence summary of what an inference  ...
An inference server is responsible for processing incoming requests and generating responses based on the input data, typically for tasks such as machine learning model inference, predictive analytics, and automated decision-making. 

PROMPT: A user asks for the weather in Riyadh and the time ...
To fetch the weather in Riyadh and the time in Tokyo for a specific date, you would typically make two API calls. Assuming the services that provide these information are accessible through two different APIs (API One for weather and API Two for time), you would issue the following calls:

For weather in Riyadh:
```plaintext
http://api.weather.com/w/chat?apiKey=[your_api_key]&conditions=all&urls=http://weather.com/re/ (replaced "re" with Riyadh)
```

To fetch the time in Tokyo (`Tokyo`):
```plaintext
http://api.timeZone.com/timetables/v1/timeships.json?key=apikey&cityIds=891&api-language=ru (Japan time zone ID 891)
```

Replace `[y

In [15]:
import os, signal, time, urllib.request, urllib.error

def shutdown_server(proc=None, port=PORT):
    try:
        proc = server if proc is None else proc
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        print(f"sent SIGTERM to process group of pid {proc.pid}")
    except (ProcessLookupError, NameError):
        print("no server process to kill")
    time.sleep(3)
    try:
        with urllib.request.urlopen(f"http://localhost:{port}/v1/models", timeout=2):
            print(f"WARNING: port {port} still answering; something is still up")
    except (urllib.error.URLError, ConnectionError, OSError):
        print(f"port {port} is free")

shutdown_server()

sent SIGTERM to process group of pid 4848
port 8000 is free


In [16]:
shutdown_server()

sent SIGTERM to process group of pid 4848
port 8000 is free


In [17]:
SERVER_ARGS_FP16 = {
    "--model": "Qwen/Qwen2.5-1.5B-Instruct",
    "--dtype": "half",
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": str(PORT),
    "--enable-auto-tool-choice": None,
    "--tool-call-parser": "hermes",
}

server = launch_server(SERVER_ARGS_FP16)

launching: /content/venv/bin/python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000 --enable-auto-tool-choice --tool-call-parser hermes
server pid 7416, logging to /content/server.log


In [18]:
healthy = wait_for_health()

server healthy after about 57s: http://localhost:8000/v1/models -> 200


In [19]:
for p in SPOT_PROMPTS:
    r = client.chat.completions.create(
        model="Qwen/Qwen2.5-1.5B-Instruct",
        messages=[{"role": "user", "content": p}], max_tokens=200)
    print("PROMPT:", p[:50], "...")
    print(r.choices[0].message.content, "\n")

PROMPT: Write a two-sentence summary of what an inference  ...
An inference server is responsible for processing incoming requests to execute model predictions and returning results to the client seamlessly and efficiently. 

PROMPT: A user asks for the weather in Riyadh and the time ...
To find both the weather in Riyadh and the time in Tokyo, you would need to use two tool calls:

1. For the weather, you would make a tool call to "WeatherForecast".
2. For the time, you would make a tool call to "TimeIn" or "Time".

Here's an example of how you might structure the JSON for these two tool calls using a hypothetical JSON web API or a chatbot API provided by the service:

```json
{
    "tool": "WeatherForecast",
    "parameters": {
        "city": "Riyadh"
    },
    "_request_id": "123456789"
},
{
    "tool": "Time",
    "parameters": {
        "location": "Tokyo",
        "number_of_decimals": 2
    },
    "_request_id": "987654321"
}
```

In this example, I've included the API key pla

In [20]:
from openai import OpenAI

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name"},
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate an arithmetic expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string",
                                   "description": "e.g. 23 * 19"},
                },
                "required": ["expression"],
            },
        },
    },
]

CANONICAL = [
    {
        "id": "two_tool",
        "k": 4,
        "wants_call": True,
        "prompt": "What is the weather in Riyadh, and what is 23 multiplied "
                  "by 19? Use your tools.",
    },
    {
        "id": "single",
        "k": 4,
        "wants_call": True,
        "prompt": "What is the weather in Tokyo right now? Use your tools.",
    },
    {
        "id": "distractor",
        "k": 2,
        "wants_call": False,
        "prompt": "In one sentence, explain what a tool call is. Do not call "
                  "any tool; just answer.",
    },
]


def _tool_calls_of(message) -> list:
    tc = getattr(message, "tool_calls", None)
    return list(tc) if tc else []


def _valid_call(call) -> bool:
    import json
    try:
        fn = call.function.name
        if fn not in ("get_weather", "calculate"):
            return False
        args = json.loads(call.function.arguments or "{}")
    except (AttributeError, ValueError):
        return False
    if fn == "get_weather":
        return isinstance(args.get("city"), str) and bool(args["city"])
    if fn == "calculate":
        return isinstance(args.get("expression"), str) and bool(args["expression"])
    return False


def run_smoke(base_url: str, model: str, temperature: float = 0.0) -> dict:
    client = OpenAI(base_url=base_url, api_key="not-needed")

    total_attempts = 0
    valid_call_attempts = 0
    distractor_attempts = 0
    distractor_call_free = 0
    per_prompt = {}

    for spec in CANONICAL:
        pid, k, wants = spec["id"], spec["k"], spec["wants_call"]
        got_valid = 0
        got_call_free = 0
        for _ in range(k):
            total_attempts += 1
            resp = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": spec["prompt"]}],
                tools=TOOLS,
                tool_choice="auto",
                temperature=temperature,
                max_tokens=256,
            )
            msg = resp.choices[0].message
            calls = _tool_calls_of(msg)
            any_valid = any(_valid_call(c) for c in calls)

            if wants:
                if any_valid:
                    valid_call_attempts += 1
                    got_valid += 1
            else:
                distractor_attempts += 1
                if not calls:
                    valid_call_attempts += 1
                    distractor_call_free += 1
                    got_call_free += 1

        per_prompt[pid] = {"k": k, "wants_call": wants,
                           "valid": got_valid, "call_free": got_call_free}

    distractor_majority = (distractor_call_free * 2 > distractor_attempts) \
        if distractor_attempts else True
    passed = (valid_call_attempts >= 8) and distractor_majority

    return {
        "model": model,
        "total_attempts": total_attempts,
        "score": valid_call_attempts,
        "distractor_attempts": distractor_attempts,
        "distractor_call_free": distractor_call_free,
        "distractor_majority_clean": distractor_majority,
        "per_prompt": per_prompt,
        "passed": passed,
    }

In [21]:
result_fp16 = run_smoke(base_url="http://localhost:8000/v1",
                        model="Qwen/Qwen2.5-1.5B-Instruct")
print(result_fp16)

{'model': 'Qwen/Qwen2.5-1.5B-Instruct', 'total_attempts': 10, 'score': 10, 'distractor_attempts': 2, 'distractor_call_free': 2, 'distractor_majority_clean': True, 'per_prompt': {'two_tool': {'k': 4, 'wants_call': True, 'valid': 4, 'call_free': 0}, 'single': {'k': 4, 'wants_call': True, 'valid': 4, 'call_free': 0}, 'distractor': {'k': 2, 'wants_call': False, 'valid': 0, 'call_free': 2}}, 'passed': True}


In [22]:
shutdown_server()

server = launch_server()
healthy = wait_for_health()

sent SIGTERM to process group of pid 7416
port 8000 is free
launching: /content/venv/bin/python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct-AWQ --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000 --quantization awq --enable-auto-tool-choice --tool-call-parser hermes
server pid 8648, logging to /content/server.log
server healthy after about 69s: http://localhost:8000/v1/models -> 200


In [23]:
result_awq = run_smoke(base_url="http://localhost:8000/v1",
                       model="Qwen/Qwen2.5-1.5B-Instruct-AWQ")
print(result_awq)

{'model': 'Qwen/Qwen2.5-1.5B-Instruct-AWQ', 'total_attempts': 10, 'score': 10, 'distractor_attempts': 2, 'distractor_call_free': 2, 'distractor_majority_clean': True, 'per_prompt': {'two_tool': {'k': 4, 'wants_call': True, 'valid': 4, 'call_free': 0}, 'single': {'k': 4, 'wants_call': True, 'valid': 4, 'call_free': 0}, 'distractor': {'k': 2, 'wants_call': False, 'valid': 0, 'call_free': 2}}, 'passed': True}


In [26]:
%%writefile model-lock.md
# Model lock (team record)

## The locked model

- Model id: `Qwen/Qwen2.5-1.5B-Instruct-AWQ`
- Quantisation: `awq`
- Why this one: passed the smoke test 10/10 with clean distractor compliance
  (identical to fp16), while freeing VRAM capacity into KV-cache (22,955 GPU
  blocks) instead of leaving it idle.

## The launch flags

Writing model-lock.md


In [28]:
%%writefile model-lock.md
# Model lock (team record)

## The locked model

- Model id: `Qwen/Qwen2.5-1.5B-Instruct-AWQ`
- Quantisation: `awq`
- Why this one: passed the smoke test 10/10 with clean distractor compliance
  (identical to fp16), while freeing VRAM capacity into KV-cache (22,955 GPU
  blocks) instead of leaving it idle.

## The launch flags

Overwriting model-lock.md


In [29]:
import json
with open("smoke_result.json", "w") as f:
    json.dump(result_awq, f, indent=2)
print("saved smoke_result.json")

saved smoke_result.json


In [30]:
shutdown_server()

sent SIGTERM to process group of pid 8648
port 8000 is free


In [31]:
import json, os, re


class _Stop(Exception):
    """Ends the check without killing the notebook kernel."""


def fail(reason: str):
    print(f"GREEN CHECK: FAIL ({reason})")
    raise _Stop()


def main() -> None:
    if not os.path.exists("smoke_result.json"):
        fail("smoke_result.json not found; write it in Cell 5")
    try:
        with open("smoke_result.json") as fh:
            result = json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"smoke_result.json is not valid JSON: {exc}")

    for key in ("score", "total_attempts", "distractor_majority_clean", "passed"):
        if key not in result:
            fail(f"smoke_result.json missing key: {key}")

    score = result["score"]
    total = result["total_attempts"]
    if not isinstance(score, int) or not isinstance(total, int):
        fail("score and total_attempts must be integers")
    if total != 10:
        fail(f"total_attempts is {total}, the smoke test defines n=10")
    if score < 8:
        fail(f"smoke score {score}/10 is below the 8/10 gate")
    if not result["distractor_majority_clean"]:
        fail("distractor did not stay call-free in the majority; a model that "
             "always calls a tool fails the real consumer")
    if not result["passed"]:
        fail("smoke test reports passed=false")

    if not os.path.exists("model-lock.md"):
        fail("model-lock.md not found")
    with open("model-lock.md") as fh:
        lock = fh.read()
    remaining = re.findall(r"FILL:", lock)
    if remaining:
        fail(f"model-lock.md has {len(remaining)} unfilled FILL: placeholders")
    if not re.search(r"Model id:\s*\S+", lock):
        fail("model-lock.md has no concrete Model id")

    print(f"smoke score: {score}/{total}, distractor clean: "
          f"{result['distractor_majority_clean']}")
    print("model-lock.md: all fields filled")
    print("GREEN CHECK: PASS")


try:
    main()
except _Stop:
    pass

smoke score: 10/10, distractor clean: True
model-lock.md: all fields filled
GREEN CHECK: PASS
